# Module 03 - Chunking and Ingestion

**Duration:** 45 minutes

Chunking is how you split documents into pieces before embedding them.
It has more impact on retrieval quality than almost any other decision in the pipeline.
This module covers the main strategies, explores the chunker built into this repo,
and walks through ingesting a set of documents into ChromaDB.

---


## 3.1 Why chunking matters

Embedding models have a token limit (typically 256 to 512 tokens).
A long document cannot be embedded as a single vector, it has to be split.

But even ignoring the token limit, chunking matters for retrieval quality:

- A chunk that is **too short** may lack enough context to match the query correctly.
- A chunk that is **too long** contains too many topics; its embedding becomes a blurry
  average that matches many things weakly rather than one thing strongly.
- **Splitting mid-sentence** or mid-paragraph destroys the meaning of the pieces.

There is no universally correct chunk size. It depends on the documents,
the embedding model, and the kinds of questions being asked.
The goal is to produce chunks that are **self-contained and focused**.

### The retrieval granularity trade-off

Chunking is fundamentally a precision vs. recall trade-off:

| Chunk size | Retrieval behaviour | Risk |
|------------|--------------------|----|
| Very small (< 50 words) | Highly specific matches | Loses surrounding context; hard to answer multi-sentence questions |
| Medium (100–300 words) | Good balance | Occasional topic bleed at boundaries |
| Large (400+ words) | Broad coverage per chunk | Embedding is diluted; many irrelevant sentences retrieved |

A common practical choice is **150-250 words** with a **10-20% overlap**,
but this should be validated against your specific documents and queries.

### Chunking strategy overview

There are four main families of chunking strategies:

1. **Fixed-size**: split every N words. Simple but crude.
2. **Sliding window**: like fixed-size, but repeat a window of text at each boundary.
3. **Structure-aware**: split on sentences, paragraphs, or section headings.
4. **Semantic**: use embeddings to detect topic shifts and split there.

We will implement 1, 2, and 3 in this module. Strategy 4 is more complex
but covered briefly in the exercises.


In [ ]:
# Read a sample document so we have something to work with
import sys

sys.path.insert(0, '..')  # only needed if running outside uv environment

from ragsst.utils import read_file

text = read_file('../data/sample_docs/aihpi-home.txt')
print(f'Document length: {len(text.split())} words')
print('\nFirst 300 characters:')
print(text[:300])


## 3.2 Strategy 1: fixed-size chunking

Split the text into chunks of exactly N words, regardless of sentence or paragraph boundaries.
Simple to implement, but the chunks often cut mid-sentence.

`split_text_basic` tokenises the entire text into a flat word list and slices it,
so every chunk is guaranteed to be within `max_words` words.


In [ ]:
from ragsst.utils import split_text_basic

chunks_basic = split_text_basic(text, max_words=100)

print(f'Number of chunks: {len(chunks_basic)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_basic[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


Notice how chunks cut mid-sentence. The text is evenly split but context is lost at boundaries.
This is the baseline we will compare all other strategies against.


## 3.3 Strategy 2: sliding window (fixed-size with overlap)

One common problem with fixed-size chunking is that relevant information
can end up split across a chunk boundary. A sentence that starts at the
end of chunk 3 and finishes at the start of chunk 4 will be poorly represented
in both chunks.

**Overlapping chunks** solve this by repeating a small window of words at each boundary.
A typical overlap is 10-20% of the chunk size (e.g. 20 words for a 100-word chunk).

The trade-off is that you store more data and may retrieve near-duplicate content.
The duplicate issue can be handled downstream (Module 05 shows deduplication in multi-query).

### Overlap vs. parent-child chunking

An alternative to simple overlap is **parent-child chunking**:
- Embed small child chunks for high-precision retrieval
- When a child chunk is retrieved, return its larger parent chunk to the LLM

This gives you the best of both worlds: precise matching AND full context.
LlamaIndex calls this "small-to-big retrieval". It is more complex to implement
but meaningfully improves answer quality for long documents.


In [ ]:
from ragsst.utils import split_text_sliding_window

chunks_sliding = split_text_sliding_window(text, max_words=100, overlap=20)

print(f'Chunks without overlap: {len(chunks_basic)}')
print(f'Chunks with overlap:    {len(chunks_sliding)}')

# Show the boundary between chunk 0 and chunk 1
print('\nEnd of chunk 0:')
print(' '.join(chunks_sliding[0].split()[-15:]))
print('\nStart of chunk 1 (overlapping words in brackets):')
print('[' + ' '.join(chunks_sliding[1].split()[:20]) + ']')
print(' '.join(chunks_sliding[1].split()[20:35]))


**Exercise:** Try `overlap=0`, `overlap=20`, and `overlap=50` with `max_words=100`.
How does the chunk count change? At what overlap does the repeated content start feeling redundant?


## 3.4 Strategy 3: structure-aware chunking

Both fixed-size strategies ignore the document's natural structure.
Structure-aware strategies split on linguistic or document boundaries instead.
This repo provides two variants:

### 3.4a Sentence-based chunking

`split_text_sentences` packs whole sentences into chunks, so a chunk never
cuts mid-sentence. An optional `overlap_sentences` parameter repeats the last
N sentences of each chunk at the start of the next, preserving local context
across boundaries.


In [ ]:
from ragsst.utils import split_text_sentences

chunks_sent = split_text_sentences(text, max_words=100, overlap_sentences=1)

print(f'Number of chunks: {len(chunks_sent)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_sent[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


### 3.4b Paragraph-based chunking

`split_text_paragraphs` packs whole paragraphs together up to `max_words`.
Works best for well-structured documents (articles, books) where paragraphs
are already coherent units of thought, because it keeps them intact rather than
splitting across arbitrary word boundaries.


In [ ]:
from ragsst.utils import split_text_paragraphs

chunks_para = split_text_paragraphs(text, max_words=100)

print(f'Number of chunks: {len(chunks_para)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_para[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


### 3.4c Heading-aware chunking

The `split_text` function treats short lines (likely headings) as section
boundaries and keeps them attached to the paragraph that follows.
This preserves section context that all previous strategies would discard.

Consider a document with this structure:

```
Refund Policy
You may return any item within 30 days for a full refund...

Shipping Policy
Standard shipping takes 3–5 business days...
```

Without heading awareness, a chunk might start with *"...30 days for a full refund"*
— which, without the heading, looks like it could be about almost anything.
With heading awareness, the chunk reads: *"Refund Policy: You may return any item..."*
and embeds much closer to queries about refunds and returns.


In [ ]:
from ragsst.utils import split_text

chunks_smart = split_text(text, max_words=100)

print(f'Number of chunks: {len(chunks_smart)}')
print('\nFirst 3 chunks:')
for i, c in enumerate(chunks_smart[:3]):
    print(f'\n--- Chunk {i+1} ({len(c.split())} words) ---')
    print(c)


### Strategy comparison

Let's compare all strategies side by side across different `max_words` settings.


In [ ]:
from ragsst.utils import (
    split_text,
    split_text_basic,
    split_text_paragraphs,
    split_text_sentences,
    split_text_sliding_window,
)

strategies = {
    'fixed-size':    lambda t, w: split_text_basic(t, max_words=w),
    'sliding (20%)': lambda t, w: split_text_sliding_window(t, max_words=w, overlap=w//5),
    'sentences':     lambda t, w: split_text_sentences(t, max_words=w),
    'paragraphs':    lambda t, w: split_text_paragraphs(t, max_words=w),
    'heading-aware': lambda t, w: split_text(t, max_words=w),
}

print(f"{'strategy':<16}  {'max=64':>8}  {'max=128':>8}  {'max=256':>8}  {'max=512':>8}")
print('-' * 58)
for name, fn in strategies.items():
    counts = [len(fn(text, w)) for w in [64, 128, 256, 512]]
    print(f'{name:<16}  {counts[0]:>8}  {counts[1]:>8}  {counts[2]:>8}  {counts[3]:>8}')


**Exercise:** Try `max_words=50` and `max_words=400` with each strategy.
Look at the resulting chunks. At what size do the chunks stop being self-contained?
Which strategy produces the most readable chunks at small sizes?


## 3.5 Document loading: what happens before chunking

Before chunking, documents must be read into plain text.
This step is called **document loading** and hides surprising complexity:

| Format | Challenge |
|--------|-----------|
| `.txt` | Easy, read directly |
| `.pdf` | May have text layer (digital) or only images (scanned) |
| `.docx` | Paragraphs, tables, headers need structure extraction |
| HTML | Must strip navigation, ads, boilerplate |
| Tables | Cell boundaries are meaningful; flattening loses structure |

The `read_file` function in `ragsst.utils` handles `.txt`, `.pdf`, and `.docx`.
For more formats, LangChain's `DocumentLoader` ecosystem covers hundreds of sources
(Notion, Confluence, email, spreadsheets, etc.) — we cover this in Bonus A.

### Metadata: what to store alongside chunks

Each chunk should be stored with metadata that helps with:
1. **Source attribution**: which document did this come from?
2. **Filtering**: only search documents from a certain date or category
3. **Deduplication**: avoid returning two chunks from the same source

In this repo, the `make_collection` method stores `{'source': filename, 'part': chunk_index}`
for each chunk. You can see this in the inspection cell below.


## 3.6 Ingesting documents into ChromaDB

Now we put it together: read documents, chunk them, embed them, and store them
in a persistent ChromaDB collection.
The `RAGTool.make_collection()` method handles all of this.


In [ ]:
from ragsst.ragtool import RAGTool

tool = RAGTool(
    data_path='../data/sample_docs',
    collection_name='workshop_docs',
)

tool.make_collection('../data/sample_docs', 'workshop_docs')
print(f'\nCollection has {tool.collection.count()} chunks.')


In [ ]:
# Inspect what was ingested
sample = tool.collection.get(limit=5, include=['documents', 'metadatas'])

print('Sample chunks from the collection:\n')
for doc, meta in zip(sample['documents'], sample['metadatas']):
    print(f"Source: {meta['source']} (part {meta['part']})")
    print(doc[:150])
    print()


In [ ]:
# Quick retrieval check
result = tool.get_relevant_text('What AI services are available?', nresults=2)
print('Retrieved context:')
print(result)


---

**Exercises**

1. Open `src/ragsst/utils.py` and read `split_text`. Trace through what happens
   when a heading line (e.g. 'AI Workshops') is encountered. Why is it kept with the next paragraph?

2. Create a new text file in `data/sample_docs/` with a few paragraphs about a topic
   you know well. Run `make_collection` again and query for something from your file.

3. Call `tool.collection.get()` without a limit and look at how many chunks
   came from each source file. Does the number of chunks seem proportional to the
   length of each document?

4. *(Advanced)* Implement a semantic chunker: embed each sentence, then split whenever
   the cosine similarity between consecutive sentence embeddings drops below a threshold.
   How does it compare to the structure-aware strategies on your sample document?

---

**Further reading**

- Chunking strategies overview: https://www.pinecone.io/learn/chunking-strategies/
- Rethinking chunk size: https://arxiv.org/abs/2505.21700
